In [1]:
# -------------------------------------------------------------
# CELL 1: SETUP AND LIBRARY IMPORTS
# -------------------------------------------------------------
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import json
import time
import os
import re

print("✅ Step 1 Complete: Automation libraries successfully loaded!")


✅ Step 1 Complete: Automation libraries successfully loaded!


In [28]:
# -------------------------------------------------------------
# CELL 2: INITIALIZE CHROME BROWSER ENGINE & AUTHENTICATE (ALERT-PROOF)
# -------------------------------------------------------------
from selenium import webdriver

options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-gpu")

# ⭐ FIX: Tell Chrome to automatically dismiss unexpected browser alerts & cookies popups
options.set_capability("unhandledPromptBehavior", "dismiss")

print("🚀 Launching alert-proof automated Chrome window instance...")
driver = webdriver.Chrome(options=options)

# Open Instagram base gate
driver.get("https://instagram.com")
print("\n🔐 Action Required:")
print("1. Go to the open Chrome automation window.")
print("2. Enter your credentials and log completely into your Instagram account.")
input("👉 Once you are logged in and looking at your feed, press ENTER here in VS Code to continue...")
print("✅ Step 2 Complete: Browser session authenticated and protected against alert errors!")


🚀 Launching alert-proof automated Chrome window instance...

🔐 Action Required:
1. Go to the open Chrome automation window.
2. Enter your credentials and log completely into your Instagram account.
✅ Step 2 Complete: Browser session authenticated and protected against alert errors!


In [20]:
# --------------------------------------------------------------------------
# CELL 3: PART 1 - SOZAL_FOODS ONLY (AUTO-SCROLL & AUTO-EXTRACT ENGINE)
# --------------------------------------------------------------------------
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import json
import time
import os
import re

csv_output_part1 = "auto_scroll_account_metrics.csv"
account_scraped_rows = []

target_account_username = "sozal_foods" 

# ⭐ FIXED: Hardcoded absolute URL completely eliminates variable string typos
print(f"📂 Navigating straight to @{target_account_username} reels feed panel...")
driver.get("https://instagram.com")
time.sleep(6)

discovered_reel_urls = set()
max_target_reels = 20
scroll_attempts = 0
max_scroll_attempts = 15

# === PHASE 1: AUTOMATICALLY SCROLL THE GRID VIEW ===
print(f"🔄 Automatically scrolling the grid layout to collect 20 reels from @{target_account_username}...")
while len(discovered_reel_urls) < max_target_reels and scroll_attempts < max_scroll_attempts:
    anchors = driver.find_elements(By.XPATH, "//a[contains(@href, '/reel/') or contains(@href, '/p/')]")
    for a in anchors:
        try:
            href = a.get_attribute("href")
            if href and "/reels/" not in href:
                # Strip tracking parameters cleanly
                clean_url = href.split("?")[0]
                discovered_reel_urls.add(clean_url)
                if len(discovered_reel_urls) >= max_target_reels:
                    break
        except: continue
            
    print(f"   🔹 Position Scrolled -> Cached {len(discovered_reel_urls)}/{max_target_reels} unique links from your profile.")
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2.5)
    scroll_attempts += 1

final_discovered_links = list(discovered_reel_urls)[:max_target_reels]
print(f"\n🎯 Discovery Done! Found {len(final_discovered_links)} items. Starting Auto-Extraction...")

# === PHASE 2: AUTOMATICALLY EXTRACT METRICS FROM EACH DETECTED REEL ===
for idx, url in enumerate(final_discovered_links, start=1):
    print(f"   🎬 Auto-Extracting Reel [{idx}/{len(final_discovered_links)}]: {url}")
    driver.get(url)
    try:
        WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.XPATH, "//main | //article")))
    except: pass
    time.sleep(5)

    row_data = {
        "Reel_Number": idx, "Reel_URL": url, "Likes": "N/A", "Comments_Count": "N/A", 
        "Views": "N/A", "Date": "N/A", "Time": "N/A", "Is_Original_Sound": "N/A"
    }

    page_source = driver.page_source

    # Extract exact posted Date and Time fields from back-end JSON scripts
    try:
        script_data = re.findall(r'<script type="application/ld\+json">(.*?)</script>', page_source)
        for script in script_data:
            parsed = json.loads(script)
            if "uploadDate" in parsed:
                raw_time = str(parsed["uploadDate"])
                if "T" in raw_time:
                    parts = raw_time.split("T")
                    row_data["Date"] = parts[0].strip()
                    row_data["Time"] = parts[1].split("+")[0].split(".")[0].strip()
                    break
    except: pass

    # Backup element tracker for timestamps
    if row_data["Date"] == "N/A":
        try:
            time_el = driver.find_element(By.XPATH, "//time")
            dt_str = str(time_el.get_attribute("datetime"))
            if dt_str and "T" in dt_str:
                parts = dt_str.split("T")
                row_data["Date"] = parts[0].strip()
                row_data["Time"] = parts[1].replace("Z", "").split(".")[0].strip()
        except: pass

    # Extract exact numbers for Likes & Comments from metadata header layers
    try:
        meta_desc = re.search(r'<meta[^>]*name=["\']description["\'][^>]*content=["\']([^"\']+)["\']', page_source)
        if meta_desc:
            desc_content = meta_desc.group(1)
            likes_match = re.search(r'([\d\.,\s?K?M?]+)\s*Likes?', desc_content, re.IGNORECASE)
            comments_match = re.search(r'([\d\.,\s?K?M?]+)\s*Comments?', desc_content, re.IGNORECASE)
            
            if likes_match: row_data["Likes"] = likes_match.group(1).strip().replace(",", "")
            if comments_match: row_data["Comments_Count"] = comments_match.group(1).strip().replace(",", "")
    except: pass

    # Local text interface parsing fallback loops
    try:
        if row_data["Likes"] == "N/A" or row_data["Comments_Count"] == "N/A":
            body_text = driver.find_element(By.TAG_NAME, "body").text
            lines = body_text.split("\n")
            for line in lines:
                cleaned_line = line.lower().strip()
                if "likes" in cleaned_line and not cleaned_line.startswith("reply") and "liked by" not in cleaned_line:
                    nums = re.findall(r'[\d\.,KkMm]+', line)
                    if nums and row_data["Likes"] == "N/A":
                        row_data["Likes"] = nums[0].replace(",", "").upper()
                if "comments" in cleaned_line and not cleaned_line.startswith("reply") and "view" not in cleaned_line:
                    nums = re.findall(r'[\d\.,KkMm]+', line)
                    if nums and row_data["Comments_Count"] == "N/A":
                        row_data["Comments_Count"] = nums[0].replace(",", "").upper()
    except: pass

    # Capture precise numerical views metrics
    try:
        view_elements = driver.find_elements(By.XPATH, "//main//*[contains(text(), 'view') or contains(text(), 'play')]")
        for el in view_elements:
            t = el.text.strip().lower()
            if any(c.isdigit() for c in t) and "reply" not in t and "facebook" not in t and len(t) < 30:
                nums = re.findall(r'[\d\.,KkMm]+', t)
                if nums:
                    row_data["Views"] = nums[0].replace(",", "").upper()
                    break
    except: pass

    # Audio Track Type Verification
    try:
        audio_txt = driver.find_element(By.XPATH, "//a[contains(@href, '/audio/')]").text.strip()
        row_data["Is_Original_Sound"] = "Yes" if "original" in audio_txt.lower() else "No"
    except: row_data["Is_Original_Sound"] = "No"

    # Print log update
    print(f"      📊 Extracted -> Likes: {row_data['Likes']} | Comments: {row_data['Comments_Count']} | Views: {row_data['Views']} | Date: {row_data['Date']} | Time: {row_data['Time']}")
    account_scraped_rows.append(row_data)

if account_scraped_rows:
    df1 = pd.DataFrame(account_scraped_rows)
    df1.to_csv(csv_output_part1, index=False, encoding="utf-8-sig")
    print(f"\n🚀 SUCCESS! Scrolled data table for @{target_account_username} saved directly inside your workspace folder as: '{csv_output_part1}'")


📂 Navigating straight to @sozal_foods reels feed panel...
🔄 Automatically scrolling the grid layout to collect 20 reels from @sozal_foods...
   🔹 Position Scrolled -> Cached 0/20 unique links from your profile.
   🔹 Position Scrolled -> Cached 0/20 unique links from your profile.
   🔹 Position Scrolled -> Cached 0/20 unique links from your profile.
   🔹 Position Scrolled -> Cached 0/20 unique links from your profile.
   🔹 Position Scrolled -> Cached 0/20 unique links from your profile.
   🔹 Position Scrolled -> Cached 0/20 unique links from your profile.
   🔹 Position Scrolled -> Cached 0/20 unique links from your profile.
   🔹 Position Scrolled -> Cached 0/20 unique links from your profile.
   🔹 Position Scrolled -> Cached 0/20 unique links from your profile.
   🔹 Position Scrolled -> Cached 0/20 unique links from your profile.
   🔹 Position Scrolled -> Cached 0/20 unique links from your profile.
   🔹 Position Scrolled -> Cached 0/20 unique links from your profile.
   🔹 Position Scrol

In [ ]:
# just auto scroll not auto extract-------------------------------------------------------------
# CELL 3: PART 1 - AUTO-SCROLL ACCOUNT DISCOVERY ENGINE
# -------------------------------------------------------------
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import json
import time
import os
import re

csv_output_part1 = "auto_scroll_account_metrics.csv"
account_scraped_rows = []

# Target user account profile handle configuration variable
target_account_username = "sozal_foods" 

# ⭐ FIXED: Fixed the variable mapping inside the print statement string layer
print(f"📂 Navigating to reels grid feed panel for profile: @{target_account_username}...")

# ⭐ FIXED: Standardized direct link format ensures zero domain syntax errors
driver.get("https://www.instagram.com/sozal_foods?utm_source=ig_web_button_share_sheet&igsh=ZDNlZDc0MzIxNw==")
time.sleep(6)

discovered_reel_urls = set()
max_target_reels = 20
scroll_attempts = 0
max_scroll_attempts = 15

print("🔄 Automatically scrolling the grid layout to collect 20 reels...")
while len(discovered_reel_urls) < max_target_reels and scroll_attempts < max_scroll_attempts:
    anchors = driver.find_elements(By.XPATH, "//a[contains(@href, '/reel/') or contains(@href, '/p/')]")
    for a in anchors:
        try:
            href = a.get_attribute("href")
            if href and "/reels/" not in href:
                clean_url = href.split("?")[0]
                discovered_reel_urls.add(clean_url)
                if len(discovered_reel_urls) >= max_target_reels:
                    break
        except: continue
            
    print(f"   🔹 Position Scrolled -> Cached {len(discovered_reel_urls)}/{max_target_reels} target URL links.")
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2.5)
    scroll_attempts += 1

final_discovered_links = list(discovered_reel_urls)[:max_target_reels]
print(f"\n🎯 Discovery Done! Found {len(final_discovered_links)} items. Auto-extracting profile metrics...")

for idx, url in enumerate(final_discovered_links, start=1):
    print(f"   🎬 Processing Reel [{idx}/{len(final_discovered_links)}]: {url}")
    driver.get(url)
    try:
        WebDriverWait(driver, 12).until(EC.presence_of_element_located((By.XPATH, "//main | //article")))
    except: pass
    time.sleep(4)

    row_data = {
        "Reel_Number": idx, "Reel_URL": url, "Likes": "N/A", "Comments_Count": "N/A", 
        "Views": "N/A", "Date": "N/A", "Time": "N/A", "Is_Original_Sound": "N/A"
    }

    try:
        page_source = driver.page_source
        script_data = re.findall(r'<script type="application/ld\+json">(.*?)</script>', page_source)
        for script in script_data:
            parsed = json.loads(script)
            if "uploadDate" in parsed:
                raw_time = parsed["uploadDate"]
                if "T" in raw_time:
                    parts = raw_time.split("T")
                    row_data["Date"] = parts[0].strip()
                    row_data["Time"] = parts[1].split("+")[0].strip()
                    break
    except: pass

    try:
        lines = driver.find_element(By.TAG_NAME, "body").text.split("\n")
        for line in lines:
            cleaned = line.lower().strip()
            if "likes" in cleaned and row_data["Likes"] == "N/A":
                val = cleaned.replace("likes", "").replace("like", "").replace(",", "").strip()
                if any(c.isdigit() for c in val): row_data["Likes"] = val.upper()
            if "comments" in cleaned and row_data["Comments_Count"] == "N/A":
                val = cleaned.replace("comments", "").replace("comment", "").replace(",", "").strip()
                if any(c.isdigit() for c in val): row_data["Comments_Count"] = val
    except: pass

    try:
        audio_txt = driver.find_element(By.XPATH, "//a[contains(@href, '/audio/')]").text.strip()
        row_data["Is_Original_Sound"] = "Yes" if "original" in audio_txt.lower() else "No"
    except: row_data["Is_Original_Sound"] = "No"

    account_scraped_rows.append(row_data)

if account_scraped_rows:
    df1 = pd.DataFrame(account_scraped_rows)
    df1.to_csv(csv_output_part1, index=False, encoding="utf-8-sig")
    print(f"\n🚀 PART 1 SUCCESSFUL! Scrolled data table saved as: '{csv_output_part1}'")


📂 Navigating to reels grid feed panel for profile: @sozal_foods...
🔄 Automatically scrolling the grid layout to collect 20 reels...
   🔹 Position Scrolled -> Cached 12/20 target URL links.
   🔹 Position Scrolled -> Cached 12/20 target URL links.
   🔹 Position Scrolled -> Cached 20/20 target URL links.

🎯 Discovery Done! Found 20 items. Auto-extracting profile metrics...
   🎬 Processing Reel [1/20]: https://www.instagram.com/sozal_foods/reel/DYLtn-CINNS/
   🎬 Processing Reel [2/20]: https://www.instagram.com/sozal_foods/reel/DZxP01DIsEA/
   🎬 Processing Reel [3/20]: https://www.instagram.com/sozal_foods/reel/DZuo705IEAL/
   🎬 Processing Reel [4/20]: https://www.instagram.com/sozal_foods/reel/DFIIF3xIzXe/
   🎬 Processing Reel [5/20]: https://www.instagram.com/sozal_foods/reel/DX4V9osIqLf/
   🎬 Processing Reel [6/20]: https://www.instagram.com/sozal_foods/reel/DNKo_RTIG4e/
   🎬 Processing Reel [7/20]: https://www.instagram.com/sozal_foods/reel/DVgklMmCAav/
   🎬 Processing Reel [8/20]: htt

In [ ]:
# final auto scroll and auto extract -------------------------------------------------------------
# CELL 3: PART 1 - HIGH-ACCURACY AUTO-SCROLL & ALL METRICS EXTRACTION
# -------------------------------------------------------------
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import json
import time
import os
import re

csv_output_part1 = "auto_scroll_account_metrics.csv"

# Safe reset: Clean out any bad file leftovers from previous runs
if os.path.exists(csv_output_part1):
    try: os.remove(csv_output_part1)
    except: pass

account_scraped_rows = []
target_account_username = "sozal_foods" 

# ⭐ FIX 1: Direct link explicitly modified to point right to the reels grid tab view
print(f"📂 Navigating straight to reels feed grid panel for profile: @{target_account_username}...")
driver.get(f"https://www.instagram.com/{target_account_username}/reels/")
time.sleep(6)

discovered_reel_urls = set()
max_target_reels = 20
scroll_attempts = 0
max_scroll_attempts = 15

print("🔄 Automatically scrolling the grid layout to collect 20 reels...")
while len(discovered_reel_urls) < max_target_reels and scroll_attempts < max_scroll_attempts:
    anchors = driver.find_elements(By.XPATH, "//a[contains(@href, '/reel/') or contains(@href, '/p/')]")
    for a in anchors:
        try:
            href = a.get_attribute("href")
            if href and "/reels/" not in href:
                clean_url = href.split("?")[0]
                discovered_reel_urls.add(clean_url)
                if len(discovered_reel_urls) >= max_target_reels:
                    break
        except: continue
            
    print(f"   🔹 Position Scrolled -> Cached {len(discovered_reel_urls)}/{max_target_reels} target URL links.")
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2.5)
    scroll_attempts += 1

final_discovered_links = list(discovered_reel_urls)[:max_target_reels]
print(f"\n🎯 Discovery Done! Found {len(final_discovered_links)} items. Starting Core Auto-Extraction...")

for idx, url in enumerate(final_discovered_links, start=1):
    print(f"   🎬 Processing Reel [{idx}/{len(final_discovered_links)}]: {url}")
    driver.get(url)
    try:
        WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.XPATH, "//main | //article")))
    except: pass
    time.sleep(5)

    row_data = {
        "Reel_Number": idx, "Reel_URL": url, "Likes": "N/A", "Comments_Count": "N/A", 
        "Views": "N/A", "Date": "N/A", "Time": "N/A", "Is_Original_Sound": "N/A"
    }

    page_source = driver.page_source

    # ⭐ FIX 2: High-Accuracy Extraction of separate Date and Time via time nodes
    try:
        time_el = driver.find_element(By.XPATH, "//time")
        datetime_str = time_el.get_attribute("datetime")  # e.g., "2026-06-25T14:33:20.000Z"
        if datetime_str and "T" in datetime_str:
            t_parts = datetime_str.split("T")
            row_data["Date"] = t_parts[0].strip()
            row_data["Time"] = t_parts[1].split(".")[0].replace("Z", "").strip()
    except: pass

    # Back-end script dictionary tag metadata fallback mapping
    try:
        if row_data["Date"] == "N/A":
            script_data = re.findall(r'<script type="application/ld\+json">(.*?)</script>', page_source)
            for script in script_data:
                parsed = json.loads(script)
                if "uploadDate" in parsed:
                    raw_time = str(parsed["uploadDate"])
                    if "T" in raw_time:
                        parts = raw_time.split("T")
                        row_data["Date"] = parts[0].strip()
                        row_data["Time"] = parts[1].split("+")[0].split(".")[0].strip()
                        break
    except: pass

    # ⭐ FIX 3: Isolate Exact Likes & Comments from unhidden open-graph meta descriptors
    try:
        meta_desc = re.search(r'<meta[^>]*name=["\']description["\'][^>]*content=["\']([^"\']+)["\']', page_source)
        if meta_desc:
            desc_content = meta_desc.group(1)
            likes_match = re.search(r'([\d\.,\s?K?M?]+)\s*Likes?', desc_content, re.IGNORECASE)
            comments_match = re.search(r'([\d\.,\s?K?M?]+)\s*Comments?', desc_content, re.IGNORECASE)
            
            if likes_match: row_data["Likes"] = likes_match[1].strip().replace(",", "")
            if comments_match: row_data["Comments_Count"] = comments_match[1].strip().replace(",", "")
    except: pass

    # Clean UI element text processing fallback checking loops
    try:
        if row_data["Likes"] == "N/A" or row_data["Comments_Count"] == "N/A":
            body_lines = driver.find_element(By.TAG_NAME, "body").text.split("\n")
            for line in body_lines:
                line_low = line.lower().strip()
                if "likes" in line_low and not line_low.startswith("reply") and "liked by" not in line_low:
                    nums = re.findall(r'[\d\.,KkMm]+', line)
                    if nums: row_data["Likes"] = nums[0].replace(",", "").upper()
                if "comments" in line_low and not line_low.startswith("reply") and "view" not in line_low:
                    nums = re.findall(r'[\d\.,KkMm]+', line)
                    if nums: row_data["Comments_Count"] = nums[0].replace(",", "").upper()
    except: pass

    # ⭐ FIX 4: Targeted Views/Plays node tracking string matcher
    try:
        view_elements = driver.find_elements(By.XPATH, "//main//*[contains(text(), 'view') or contains(text(), 'play')] | //span[contains(text(), 'plays')]")
        for el in view_elements:
            t = el.text.strip().lower()
            if any(c.isdigit() for c in t) and "reply" not in t and "facebook" not in t and len(t) < 35:
                nums = re.findall(r'[\d\.,KkMm]+', t)
                if nums:
                    row_data["Views"] = nums[0].replace(",", "").upper()
                    break
    except: pass

    # Audio Track Verification
    try:
        audio_txt = driver.find_element(By.XPATH, "//a[contains(@href, '/audio/')]").text.strip()
        row_data["Is_Original_Sound"] = "Yes" if "original" in audio_txt.lower() else "No"
    except: row_data["Is_Original_Sound"] = "No"

    print(f"      📊 Extracted -> Likes: {row_data['Likes']} | Comments: {row_data['Comments_Count']} | Views: {row_data['Views']} | Date: {row_data['Date']} | Time: {row_data['Time']}")
    account_scraped_rows.append(row_data)

if account_scraped_rows:
    df1 = pd.DataFrame(account_scraped_rows)
    df1.to_csv(csv_output_part1, index=False, encoding="utf-8-sig")
    print(f"\n🚀 PART 1 SUCCESSFUL! Scrolled data table saved cleanly as: '{csv_output_part1}'")


📂 Navigating straight to reels feed grid panel for profile: @sozal_foods...
🔄 Automatically scrolling the grid layout to collect 20 reels...
   🔹 Position Scrolled -> Cached 12/20 target URL links.
   🔹 Position Scrolled -> Cached 12/20 target URL links.
   🔹 Position Scrolled -> Cached 20/20 target URL links.

🎯 Discovery Done! Found 20 items. Starting Core Auto-Extraction...
   🎬 Processing Reel [1/20]: https://www.instagram.com/sozal_foods/reel/DYLtn-CINNS/
      📊 Extracted -> Likes: 2760 | Comments:  47 | Views: N/A | Date: 2026-05-11 | Time: 15:14:23
   🎬 Processing Reel [2/20]: https://www.instagram.com/sozal_foods/reel/DVGwmPxjI8O/
      📊 Extracted -> Likes: 3623 | Comments:  29 | Views: N/A | Date: 2026-02-23 | Time: 15:40:55
   🎬 Processing Reel [3/20]: https://www.instagram.com/sozal_foods/reel/DZxP01DIsEA/
      📊 Extracted -> Likes: 94 | Comments:  7 | Views: N/A | Date: 2026-06-19 | Time: 13:45:41
   🎬 Processing Reel [4/20]: https://www.instagram.com/sozal_foods/reel/DZ

In [29]:
# -------------------------------------------------------------
# CELL 4: PART 2 - FIXED LINK DATASET METRICS EXTRACTOR
# -------------------------------------------------------------
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import json
import time
import os
import re

csv_output_part2 = "provided_links_metrics.csv"

# Safe wipe: Clear any old files before writing fresh table data
if os.path.exists(csv_output_part2):
    try: os.remove(csv_output_part2)
    except: pass

fixed_link_rows = []

# Your exact list of links with tracking parameters perfectly formatted
provided_target_links = [
    "https://www.instagram.com/reel/DNKo_RTIG4e/",
    "https://www.instagram.com/reel/DEAQGXgolTr/",
    "https://www.instagram.com/p/DEP6qhuoE8P/",
    "https://www.instagram.com/reel/DE7o_rBolMe/",
    "https://www.instagram.com/reel/DFnAh5HImd3/",
    "https://www.instagram.com/p/DG0HCu9ozKR/",
    "https://www.instagram.com/reel/DHDplYQonBG/",
    "https://www.instagram.com/reel/DHqT8e0IoCC/",
    "https://www.instagram.com/reel/DIYpQU4IX0r/",
    "https://www.instagram.com/reel/DJd-qqUIPFT/",
    "https://www.instagram.com/reel/DKtoCG6I4LT/",
    "https://www.instagram.com/reel/DLJ9-WbIUN_/",
    "https://www.instagram.com/reel/DLzTzT0oTrB/",
    "https://www.instagram.com/reel/DMkMpagMMmW/",
    "https://www.instagram.com/reel/DOQegflCBSx/",
    "https://www.instagram.com/reel/DP6YYJ2iLDi/",
    "https://www.instagram.com/reel/DSzwhp_CBJQ/",
    "https://www.instagram.com/reel/DUN1ZoiCILT/",
    "https://www.instagram.com/reel/DVGwmPxjI8O/",
    "https://www.instagram.com/reel/DYFDCQVIjSF/",
    "https://www.instagram.com/reel/DYLtn-CINNS/"
]

print(f"🔄 Processing and parsing metadata from {len(provided_target_links)} provided links...")

for index, url in enumerate(provided_target_links, start=1):
    print(f"   📊 Pulling Metrics Table for Link [{index}/{len(provided_target_links)}]: {url}")
    driver.get(url)
    try:
        WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.XPATH, "//main | //article")))
    except: pass
    time.sleep(5)

    row_entry = {
        "Reel_Number": index, "Reel_URL": url, "Likes": "N/A", "Comments_Count": "N/A", 
        "Views": "N/A", "Share": "Hidden", "Repost": "Hidden", "Date": "N/A", "Time": "N/A", 
        "Sound_Name": "N/A", "Is_Original_Sound": "N/A"
    }

    page_source = driver.page_source

    # 1. High-Accuracy Extraction of separate Date and Time via time nodes
    try:
        time_el = driver.find_element(By.XPATH, "//time")
        datetime_str = time_el.get_attribute("datetime")  # e.g., "2026-06-25T14:33:20.000Z"
        if datetime_str and "T" in datetime_str:
            t_parts = datetime_str.split("T")
            row_entry["Date"] = t_parts[0].strip()
            row_entry["Time"] = t_parts[1].split(".")[0].replace("Z", "").strip()
    except: pass

    # Back-end JSON data fallback tracking
    try:
        if row_entry["Date"] == "N/A":
            script_data = re.findall(r'<script type="application/ld\+json">(.*?)</script>', page_source)
            for data_slice in script_data:
                parsed_json = json.loads(data_slice)
                if "uploadDate" in parsed_json:
                    time_stamp = str(parsed_json["uploadDate"])
                    if "T" in time_stamp:
                        ts_parts = time_stamp.split("T")
                        row_entry["Date"] = ts_parts[0].strip()
                        row_entry["Time"] = ts_parts[1].split("+")[0].split(".")[0].strip()
                        break
    except: pass

    # 2. Extract exact Likes & Comments from unhidden open-graph meta descriptors
    try:
        meta_desc = re.search(r'<meta[^>]*name=["\']description["\'][^>]*content=["\']([^"\']+)["\']', page_source)
        if meta_desc:
            desc_content = meta_desc.group(1)
            likes_match = re.search(r'([\d\.,\s?K?M?]+)\s*Likes?', desc_content, re.IGNORECASE)
            comments_match = re.search(r'([\d\.,\s?K?M?]+)\s*Comments?', desc_content, re.IGNORECASE)
            
            if likes_match: row_entry["Likes"] = likes_match.group(1).strip().replace(",", "")
            if comments_match: row_entry["Comments_Count"] = comments_match.group(1).strip().replace(",", "")
    except: pass

    # UI Element text fallback scan loops
    try:
        if row_entry["Likes"] == "N/A" or row_entry["Comments_Count"] == "N/A":
            body_lines = driver.find_element(By.TAG_NAME, "body").text.split("\n")
            for line in body_lines:
                line_low = line.lower().strip()
                if "likes" in line_low and not line_low.startswith("reply") and "liked by" not in line_low:
                    nums = re.findall(r'[\d\.,KkMm]+', line)
                    if nums: row_entry["Likes"] = nums[0].replace(",", "").upper()
                if "comments" in line_low and not line_low.startswith("reply") and "view" not in line_low:
                    nums = re.findall(r'[\d\.,KkMm]+', line)
                    if nums: row_entry["Comments_Count"] = nums[0].replace(",", "").upper()
    except: pass

    # 3. Targeted Views/Plays node tracking string matcher
    try:
        view_elements = driver.find_elements(By.XPATH, "//main//*[contains(text(), 'view') or contains(text(), 'play')] | //span[contains(text(), 'plays')]")
        for el in view_elements:
            t = el.text.strip().lower()
            if any(c.isdigit() for c in t) and "reply" not in t and "facebook" not in t and len(t) < 35:
                nums = re.findall(r'[\d\.,KkMm]+', t)
                if nums:
                    row_entry["Views"] = nums[0].replace(",", "").upper()
                    break
    except: pass

    # 4. Audio Track Extraction
    try:
        audio_link_text = driver.find_element(By.XPATH, "//a[contains(@href, '/audio/')]").text.strip()
        row_entry["Sound_Name"] = audio_link_text.replace(",", "")
        row_entry["Is_Original_Sound"] = "Yes" if "original" in audio_link_text.lower() else "No"
    except:
        row_entry["Is_Original_Sound"] = "No"

    print(f"      📊 Saved -> Likes: {row_entry['Likes']} | Comments: {row_entry['Comments_Count']} | Views: {row_entry['Views']} | Date: {row_entry['Date']} | Time: {row_entry['Time']}")
    fixed_link_rows.append(row_entry)

# Save to your second required spreadsheet file
if fixed_link_rows:
    df2 = pd.DataFrame(fixed_link_rows)
    df2.to_csv(csv_output_part2, index=False, encoding="utf-8-sig")
    print(f"\n🚀 PART 2 COMPLETE! Verified data saved cleanly to your folder as: '{csv_output_part2}'")
    print("\n📋 PRESENTATION MATRIX DATA TABLE DISPLAY:")
    print("=" * 110)
    print(df2[["Reel_Number", "Likes", "Comments_Count", "Views", "Date", "Time", "Is_Original_Sound"]].to_string(index=False))
    print("=" * 110)


🔄 Processing and parsing metadata from 21 provided links...
   📊 Pulling Metrics Table for Link [1/21]: https://www.instagram.com/reel/DNKo_RTIG4e/
      📊 Saved -> Likes: 44K | Comments:  332 | Views: N/A | Date: N/A | Time: N/A
   📊 Pulling Metrics Table for Link [2/21]: https://www.instagram.com/reel/DEAQGXgolTr/
      📊 Saved -> Likes: 279 | Comments:  4 | Views: N/A | Date: N/A | Time: N/A
   📊 Pulling Metrics Table for Link [3/21]: https://www.instagram.com/p/DEP6qhuoE8P/
      📊 Saved -> Likes: N/A | Comments: N/A | Views: N/A | Date: 2024-12-31 | Time: 16:01:32
   📊 Pulling Metrics Table for Link [4/21]: https://www.instagram.com/reel/DE7o_rBolMe/
      📊 Saved -> Likes: 252 | Comments:  11 | Views: N/A | Date: N/A | Time: N/A
   📊 Pulling Metrics Table for Link [5/21]: https://www.instagram.com/reel/DFnAh5HImd3/
      📊 Saved -> Likes: 454 | Comments:  15 | Views: N/A | Date: N/A | Time: N/A
   📊 Pulling Metrics Table for Link [6/21]: https://www.instagram.com/p/DG0HCu9ozKR/
 

In [34]:
# -------------------------------------------------------------
# CELL 9 - DEEP COMMENT SCRAPER (100% ACCURATE LINKS & STRINGS)
# -------------------------------------------------------------
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time
import os

# Initialize self-contained browser profile arguments
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-gpu")

print("🚀 Initializing precision Chrome browser session...")
driver = webdriver.Chrome(options=options)

# Your exact links including all url meta parameters from the file
comment_target_links = [
    "https://www.instagram.com/reel/DNKo_RTIG4e/",
    "https://www.instagram.com/reel/DEAQGXgolTr/",
    "https://www.instagram.com/p/DEP6qhuoE8P/",
    "https://www.instagram.com/reel/DE7o_rBolMe/",
    "https://www.instagram.com/reel/DFnAh5HImd3/",
    "https://www.instagram.com/p/DG0HCu9ozKR/",
    "https://www.instagram.com/reel/DHDplYQonBG/",
    "https://www.instagram.com/reel/DHqT8e0IoCC/",
    "https://www.instagram.com/reel/DIYpQU4IX0r/",
    "https://www.instagram.com/reel/DJd-qqUIPFT/",
    "https://www.instagram.com/reel/DKtoCG6I4LT/",
    "https://www.instagram.com/reel/DLJ9-WbIUN_/",
    "https://www.instagram.com/reel/DLzTzT0oTrB/",
    "https://www.instagram.com/reel/DMkMpagMMmW/",
    "https://www.instagram.com/reel/DOQegflCBSx/",
    "https://www.instagram.com/reel/DP6YYJ2iLDi/",
    "https://www.instagram.com/reel/DSzwhp_CBJQ/",
    "https://www.instagram.com/reel/DUN1ZoiCILT/",
    "https://www.instagram.com/reel/DVGwmPxjI8O/",
    "https://www.instagram.com/reel/DYFDCQVIjSF/",
    "https://www.instagram.com/reel/DYLtn-CINNS/"
]
comments_csv_destination = "auto_deep_comments_output.csv"
extracted_comment_rows = []

# Synchronize login data structure wall
print("🔐 Opening baseline interface platform...")
driver.get("https://www.instagram.com")
print("⏳ CRITICAL: Provide credentials and log into Instagram inside the open window now.")
input("👉 Once fully authenticated inside the browser, press ENTER here to launch scraping...")

print(f"\n🔄 Executing clean parsing loop for {len(comment_target_links)} links...")

for index, url in enumerate(comment_target_links, start=1):
    print(f"\n💬 [{index}/{len(comment_target_links)}] Accessing URL: {url}")
    driver.get(url)
    
    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.XPATH, "//main | //div[@role='main'] | //article"))
        )
    except:
        pass
    time.sleep(5)  # Render safety window

    # A. Target and trigger hidden comment feed expansion sheets Safely
    try:
        view_all_btn = driver.find_element(By.XPATH, "//*[contains(text(), 'View all') or contains(text(), 'comments') or contains(text(), 'See comments')]")
        # ⭐ FIX: Changed 'arguments.click()' to method call format 'arguments[0].click()' to prevent JS runtime exceptions
        driver.execute_script("arguments[0].click();", view_all_btn)
        time.sleep(3)
    except:
        pass

    # B. Interactive Scroll Box Component Target loop
    try:
        scroll_box = driver.find_element(By.XPATH, "//div[@role='dialog']//ul | //ul[contains(@class, '_a9zm')]")
        for _ in range(4):  
            driver.execute_script("arguments[0].scrollTo(0, arguments[0].scrollHeight);", scroll_box)
            time.sleep(2)
    except:
        for _ in range(3):
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)

    # C. Accuracy Target Mapping (Extracts pure text bodies, ignoring usernames and counts)
    try:
        comment_nodes = driver.find_elements(By.XPATH, "//main//div[@dir='auto'] | //article//div[@dir='auto'] | //div[@role='dialog']//div[@dir='auto']")
        
        seen_texts = set()
        current_reel_count = 0

        for node in comment_nodes:
            try:
                txt = node.text.strip()
                
                # Validation filter metrics
                if len(txt) > 3 and "reply" not in txt.lower() and "translation" not in txt.lower():
                    # Strip interface and user metric metadata blocks
                    if not any(m in txt.lower() for m in ["likes", "liked by", "others", "view replies", "©", "privacy", "terms", "about"]):
                        # Strip standard timeline stamps (1w, 3d, 4h, etc.)
                        if not (txt.endswith('w') or txt.endswith('d') or txt.endswith('h') or txt.endswith('m') or txt.isdigit()):
                            if txt not in seen_texts and len(txt) < 400:
                                seen_texts.add(txt)
                                current_reel_count += 1
                                
                                extracted_comment_rows.append({
                                    "reel_url": url,
                                    "comment_number": current_reel_count,
                                    "comment_text": txt
                                })
            except Exception:
                continue # Catch stale elements inline automatically
                
        print(f"   📊 Collected {current_reel_count} clean user text comments.")
    except Exception as e:
        print(f"   ⚠️ Parsing structure anomaly: {e}")

# -------------------------------------------------------------
# INDEPENDENT CLEAN REWRITE PIPELINE
# -------------------------------------------------------------
if extracted_comment_rows:
    df_comments = pd.DataFrame(extracted_comment_rows)
    df_comments.to_csv(
        comments_csv_destination,
        mode='w',  # Clears any previous dirty attempts and writes completely fresh records
        index=False,
        encoding="utf-8-sig"
    )
    print(f"\n🚀 Complete matrix processing finalized! Saved to: '{comments_csv_destination}'")
else:
    print("\n🛑 Loop ended. No content passed layout parsing targets.")



🚀 Initializing precision Chrome browser session...
🔐 Opening baseline interface platform...
⏳ CRITICAL: Provide credentials and log into Instagram inside the open window now.

🔄 Executing clean parsing loop for 21 links...

💬 [1/21] Accessing URL: https://www.instagram.com/reel/DNKo_RTIG4e/
   📊 Collected 11 clean user text comments.

💬 [2/21] Accessing URL: https://www.instagram.com/reel/DEAQGXgolTr/
   📊 Collected 12 clean user text comments.

💬 [3/21] Accessing URL: https://www.instagram.com/p/DEP6qhuoE8P/
   📊 Collected 2 clean user text comments.

💬 [4/21] Accessing URL: https://www.instagram.com/reel/DE7o_rBolMe/
   📊 Collected 12 clean user text comments.

💬 [5/21] Accessing URL: https://www.instagram.com/reel/DFnAh5HImd3/
   📊 Collected 10 clean user text comments.

💬 [6/21] Accessing URL: https://www.instagram.com/p/DG0HCu9ozKR/
   📊 Collected 2 clean user text comments.

💬 [7/21] Accessing URL: https://www.instagram.com/reel/DHDplYQonBG/
   📊 Collected 7 clean user text comme

In [36]:
# -------------------------------------------------------------
# CELL 10 - STRICT 3-CATEGORY SENTIMENT ENGINE
# -------------------------------------------------------------
import pandas as pd
import os
import re

input_csv = "auto_deep_comments_output.csv"
analyzed_csv = "analyzed_comments_dataset.csv"

if not os.path.isfile(input_csv):
    print(f"❌ Error: Cannot find '{input_csv}' in your project directory. Please check the sidebar.")
else:
    # Load dataset
    df = pd.read_csv(input_csv)
    print(f"📊 Successfully loaded {len(df)} extracted user comments for strict sentiment mapping...")

    def categorize_comment(comment):
        text = str(comment).strip()
        text_lower = text.lower()
        
        # 1. STRATEGIC NEGATIVE MAPPING
        # Combines critiques, disappointment markers, and frustrated emojis
        negative_markers = [
            "bad", "worst", "disgusting", "ruin", "waste", "unhealthy", "bakwas", 
            "ganda", "not good", "israeli", "boycott", "fail", "🤮", "🤢", "❌"
        ]
        if any(marker in text_lower for marker in negative_markers):
            return "Negative"
            
        # 2. STRATEGIC POSITIVE MAPPING
        # Combines support praise, engagement validation, and appreciation emojis
        positive_markers = [
            "mashaallah", "mashallah", "love", "wow", "yum", "yummy", "delicious", 
            "congrats", "congratulations", "proud", "keep growing", "best", "perfect", 
            "❤️", "🥰", "😍", "🔥", "👍", "😂", "🤣", "haha", "hahaha", "lol"
        ]
        if any(marker in text_lower for marker in positive_markers):
            return "Positive"
            
        # 3. NEUTRAL FALLBACK
        # Captures informational strings, queries, and standard plain statements
        return "Neutral"

    # Execute conversion pipeline
    df["predicted_sentiment"] = df["comment_text"].apply(categorize_comment)

    # Save cleanly to your structured output dataset
    df.to_csv(analyzed_csv, index=False, encoding="utf-8-sig")
    
    print("\n✅ Strict 3-category sentiment classification processing complete!")
    print(f"💾 Fresh analytical sheet saved to: '{analyzed_csv}'\n")
    
    # Render final presentation summary table in terminal
    print("📊 DISTRIBUTION ANALYSIS MATRIX SUMMARY:")
    print("=" * 50)
    distribution = df["predicted_sentiment"].value_counts()
    for category, count in distribution.items():
        percentage = (count / len(df)) * 100
        print(f"   🔹 {category.ljust(12)} : {str(count).rjust(4)} rows ({percentage:.1f}%)")
    print("=" * 50)


📊 Successfully loaded 217 extracted user comments for strict sentiment mapping...

✅ Strict 3-category sentiment classification processing complete!
💾 Fresh analytical sheet saved to: 'analyzed_comments_dataset.csv'

📊 DISTRIBUTION ANALYSIS MATRIX SUMMARY:
   🔹 Neutral      :  146 rows (67.3%)
   🔹 Positive     :   68 rows (31.3%)
   🔹 Negative     :    3 rows (1.4%)
